# Markdown 分片练习

这个 Notebook 从上到下完成：定位本地中英文原文、观察结构、解析小节、分片自检、校验来源并写出本地片段。每个代码单元前的中文注释说明作用。

教材原文在被 Git 忽略的 data/raw/；生成的 data/index/chunks.jsonl 也只保留在本地。提交 Notebook 前请清除含教材正文的运行输出。这里用字符长度演示分片；接入具体 embedding/生成模型后，还要按该模型的 tokenizer 校验 token 上限。


In [ ]:
# 本段作用：定位项目根目录，并确认中英文 Transformer 原文已下载到本地。
from pathlib import Path

current = Path.cwd().resolve()
repo_root = next((path for path in (current, *current.parents) if (path / 'config' / 'sources.yaml').is_file()), None)
if repo_root is None:
    raise FileNotFoundError('请从项目目录或其子目录启动 Notebook')

sources = {
    'zh': repo_root / 'data/raw/d2l-zh/chapter_attention-mechanisms/transformer.md',
    'en': repo_root / 'data/raw/d2l-en/chapter_attention-mechanisms-and-transformers/transformer.md',
}
print('项目目录:', repo_root)
for language, path in sources.items():
    print(f'{language}: 存在={path.is_file()}, 路径={path.relative_to(repo_root)}')
    if not path.is_file():
        raise FileNotFoundError(f'缺少本地教材文件: {path}')


## 第 1 步：观察原文

先查看中文文件的开头，注意标题、`:label:` 锚点，以及可能出现的 D2L 执行代码块。把 `language` 改成 `'en'` 可以查看英文文件。

In [ ]:
# 本段作用：先观察一篇原文的前 20 行；改为 en 可查看英文。
language = 'zh'
source_path = sources[language]
lines = source_path.read_text(encoding='utf-8').splitlines()
print(f'{language}: {len(lines)} 行；前 20 行：')
for number, line in enumerate(lines[:20], start=1):
    print(f'{number:>4}: {line}')


In [ ]:
# 本段作用：做一次快速结构扫描，便于看到标题和锚点的原始格式。
# 注意：这一步没有识别代码围栏，代码中的 # 可能误报；下方解析器会修正。
import re

heading_pattern = re.compile(r'^(#{1,6})[ \t]+(.+)')
structure = []
for number, line in enumerate(lines, start=1):
    heading = heading_pattern.match(line)
    if heading:
        structure.append((number, f'H{len(heading.group(1))}', heading.group(2)))
    elif line.startswith(':label:'):
        structure.append((number, '锚点', line))

print(f'粗略扫描到 {len(structure)} 个标题或锚点；前 30 个：')
for number, kind, value in structure[:30]:
    print(f'{number:>4}  {kind:<4} {value}')


In [ ]:
# 本段作用：识别真正的 Markdown 标题，同时避开代码围栏中的 Python 注释。
import re

HEADING_RE = re.compile(r"^(#{1,6})[ \t]+(.+?)\s*$")
FENCE_RE = re.compile(r"^\s*(`{3,}|~{3,})")
LABEL_RE = re.compile(r"^:label:`([^`]+)`$")
BUILD_DIRECTIVE_RE = re.compile(r"^:(?:label|width|height):")
CODE_BUILD_RE = re.compile(r"^\s*(?:#@(?:tab|save)\b|%load_ext d2lbook\.tab|tab\.interact_select\()")


def parse_sections(markdown: str) -> list[dict]:
    """按标题得到小节，并保留正文原始行号。"""
    sections = []
    heading_stack = []
    current_section = None
    fence_character = None
    fence_length = 0

    for line_number, line in enumerate(markdown.splitlines(), start=1):
        # 围栏开启与结束都记录在正文中；围栏里的 # 不参与标题识别。
        fence = FENCE_RE.match(line)
        if fence:
            marker = fence.group(1)
            if fence_character is None:
                fence_character, fence_length = marker[0], len(marker)
            elif marker[0] == fence_character and len(marker) >= fence_length:
                fence_character, fence_length = None, 0
            if current_section is not None:
                current_section["body_lines"].append((line_number, line))
            continue
        if fence_character is not None:
            # D2L 的构建标记不参与知识检索；普通代码和注释仍保留。
            if current_section is not None and not CODE_BUILD_RE.match(line):
                # 行尾的构建标记只删标记，保留函数定义等有效代码。
                clean_line = re.sub(r"[ \t]+#@(?:tab|save)\b.*$", "", line)
                current_section["body_lines"].append((line_number, clean_line))
            continue

        heading = HEADING_RE.match(line)
        if heading:
            if current_section is not None:
                sections.append(current_section)
            level = len(heading.group(1))
            heading_stack = heading_stack[:level - 1]
            heading_stack.append(heading.group(2).strip())
            current_section = {
                "section_path": tuple(heading_stack),
                "heading_line": line_number,
                "anchor": None,
                "body_lines": [],
            }
            continue

        # 紧随标题的章节锚点保存在元数据；图片宽度和图注锚点不进入检索正文。
        if current_section is None:
            continue
        label = LABEL_RE.match(line.strip())
        if label and current_section["anchor"] is None and not any(
            body.strip() for _, body in current_section["body_lines"]
        ):
            current_section["anchor"] = label.group(1)
            continue
        if BUILD_DIRECTIVE_RE.match(line.strip()):
            continue
        current_section["body_lines"].append((line_number, line))

    if current_section is not None:
        sections.append(current_section)
    return sections

In [ ]:
# 检查解析器是否把代码中的 # 注释误认为 Markdown 标题。
def test_ignore_headings_inside_code_fences():
    label = ":label:" + chr(96) + "sec_attention" + chr(96)
    fixture = f"""~~~python
# 伪标题
print('准备环境')
~~~
# 注意力
{label}

第一段正文。

~~~python
# 代码中的伪标题
print('示例代码')
~~~

## 细节
第二段正文。
"""
    sections = parse_sections(fixture)
    assert [item["section_path"] for item in sections] == [
        ("注意力",),
        ("注意力", "细节"),
    ]
    assert sections[0]["anchor"] == "sec_attention"
    assert any("# 代码中的伪标题" in line for _, line in sections[0]["body_lines"])

test_ignore_headings_inside_code_fences()
print("通过：代码块中的 # 不会产生伪章节。")

In [ ]:
# 本段作用：先按段落、代码块和公式块划分；代码与公式保持完整。
import hashlib


def section_blocks(section: dict) -> list[dict]:
    """把小节正文拆成带原始行号的块。"""
    blocks = []
    prose = []
    special = []
    mode = None
    fence_character = None
    fence_length = 0

    def flush_prose():
        if prose:
            blocks.append({
                "text": "\n".join(text for _, text in prose).strip(),
                "line_start": prose[0][0],
                "line_end": prose[-1][0],
                "atomic": False,
            })
            prose.clear()

    def flush_special():
        if special:
            blocks.append({
                "text": "\n".join(text for _, text in special).strip(),
                "line_start": special[0][0],
                "line_end": special[-1][0],
                "atomic": True,
            })
            special.clear()

    for line_number, line in section["body_lines"]:
        fence = FENCE_RE.match(line)
        if mode == "fence":
            special.append((line_number, line))
            if fence and fence.group(1)[0] == fence_character and len(fence.group(1)) >= fence_length:
                flush_special()
                mode = None
            continue
        if mode == "math":
            special.append((line_number, line))
            if line.strip().endswith("$$"):
                flush_special()
                mode = None
            continue
        if fence:
            flush_prose()
            marker = fence.group(1)
            fence_character, fence_length = marker[0], len(marker)
            special.append((line_number, line))
            mode = "fence"
        elif line.strip().startswith("$$"):
            flush_prose()
            special.append((line_number, line))
            if not (line.strip().endswith("$$") and len(line.strip()) > 2):
                mode = "math"
            else:
                flush_special()
        elif not line.strip():
            flush_prose()
        else:
            prose.append((line_number, line))
    flush_prose()
    flush_special()
    return [block for block in blocks if block["text"]]


def split_long_prose(block: dict, max_chars: int, overlap_chars: int) -> list[dict]:
    """长段落优先在句号/空白处切；相邻片段重叠少量字符。"""
    text = block["text"]
    if block["atomic"] or len(text) <= max_chars:
        return [block]
    pieces = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            # 尽量从窗口后半段寻找自然断点；没有则按长度截断。
            tail = text[start + max_chars // 2:end]
            boundaries = list(re.finditer(r"[。！？.!?；;]\s*|\s+", tail))
            if boundaries:
                end = start + max_chars // 2 + boundaries[-1].end()
        part = text[start:end].strip()
        if part:
            left = text.find(part, start, end)
            line_start = block["line_start"] + text[:left].count("\n")
            line_end = block["line_start"] + text[:left + len(part)].count("\n")
            pieces.append({
                "text": part,
                "line_start": line_start,
                "line_end": line_end,
                "atomic": False,
            })
        if end == len(text):
            break
        # 重叠只用于长段落；代码块和公式块从不在内部切开。
        start = max(start + 1, end - overlap_chars)
    return pieces


def chunk_markdown(markdown: str, source: dict, max_chars: int = 1200,
                   overlap_chars: int = 120) -> list[dict]:
    """生成稳定的、可追溯到固定 GitHub 提交和原文行号的片段。"""
    if max_chars < 40 or not 0 <= overlap_chars < max_chars // 2:
        raise ValueError("要求 max_chars >= 40，且 0 <= overlap_chars < max_chars / 2")
    required = {"source_id", "repo", "revision", "path", "language"}
    if not required <= source.keys():
        raise ValueError(f"缺少来源字段：{sorted(required - source.keys())}")
    document_id = hashlib.sha256(source["source_id"].encode("utf-8")).hexdigest()[:16]
    chunks = []

    for section in parse_sections(markdown):
        blocks = [
            piece
            for block in section_blocks(section)
            for piece in split_long_prose(block, max_chars, overlap_chars)
        ]
        # 同一小节内合并短块；遇到长块则单独成片。目标长度对不可拆的代码/公式是软上限。
        groups = []
        group = []
        size = 0
        for block in blocks:
            addition = len(block["text"]) + (2 if group else 0)
            if group and size + addition > max_chars:
                groups.append(group)
                group, size = [], 0
            group.append(block)
            size += len(block["text"]) + (2 if len(group) > 1 else 0)
        if group:
            groups.append(group)

        for group in groups:
            body = "\n\n".join(block["text"] for block in group)
            line_start = group[0]["line_start"]
            line_end = group[-1]["line_end"]
            section_name = " / ".join(section["section_path"])
            identity = "\n".join((
                source["source_id"],
                source["path"],
                section["anchor"] or section_name,
                str(line_start),
                body,
            ))
            chunk_id = hashlib.sha256(identity.encode("utf-8")).hexdigest()[:24]
            url = (
                f'https://github.com/{source["repo"]}/blob/{source["revision"]}/'
                f'{source["path"]}#L{line_start}-L{line_end}'
            )
            chunks.append({
                "chunk_id": chunk_id,
                "source_id": source["source_id"],
                "document_id": document_id,
                "section": section_name,
                "text": body,
                "language": source["language"],
                "source_path": source["path"],
                "line_start": line_start,
                "line_end": line_end,
                "url": url,
            })
    return chunks


In [ ]:
# 本段作用：用自写小文档验证长段落切分、稳定 ID 和原文链接。
def test_chunking_and_provenance():
    label = ":label:" + chr(96) + "sec_demo" + chr(96)
    fixture = "# 正文\n" + label + "\n\n" + "第一句说明。" * 32 + """

## 代码
~~~python
#@tab pytorch
#@save
# 不是标题
def f():  #@save
    return 1
print(1)
~~~
"""
    source = {
        "source_id": "demo:chapter/demo.md",
        "repo": "example/demo",
        "revision": "a" * 40,
        "path": "chapter/demo.md",
        "language": "zh",
    }
    chunks = chunk_markdown(fixture, source, max_chars=120, overlap_chars=12)
    assert len(chunks) >= 3
    assert chunks == chunk_markdown(fixture, source, max_chars=120, overlap_chars=12)
    assert len({item["chunk_id"] for item in chunks}) == len(chunks)
    assert all(item["url"].startswith("https://github.com/example/demo/blob/" + "a" * 40)
               for item in chunks)
    assert all(item["line_start"] <= item["line_end"] for item in chunks)
    code_chunks = [item for item in chunks if "# 不是标题" in item["text"]]
    assert len(code_chunks) == 1
    assert "~~~python" in code_chunks[0]["text"] and "~~~" in code_chunks[0]["text"]
    assert "#@tab" not in code_chunks[0]["text"] and "#@save" not in code_chunks[0]["text"]
    assert "def f():" in code_chunks[0]["text"]
    assert not any(item["section"].endswith("不是标题") for item in chunks)

test_chunking_and_provenance()
print("通过：长段落分片、代码块、稳定 ID 和出处链接。")

In [ ]:
# 本段作用：核验 28 篇本地中英文原文的哈希，分片后写入被 Git 忽略的 JSONL。
import json
from collections import Counter

manifest_path = repo_root / "data/raw/manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
all_chunks = []
for item in manifest:
    repo_name = item["repo"].split("/")[-1]
    local_path = repo_root / "data/raw" / repo_name / item["path"]
    raw_bytes = local_path.read_bytes()
    actual_sha = hashlib.sha256(raw_bytes).hexdigest()
    if actual_sha != item["sha256"] or len(raw_bytes) != item["bytes"]:
        raise ValueError(f"文件校验失败：{local_path.relative_to(repo_root)}")
    source = {
        "source_id": f'{repo_name}:{item["path"]}',
        "repo": item["repo"],
        "revision": item["revision"],
        "path": item["path"],
        "language": item["language"],
    }
    # 用已校验的 UTF-8 原文分片；改这两个参数可以比较粒度与重叠。
    all_chunks.extend(chunk_markdown(
        raw_bytes.decode("utf-8"), source, max_chars=1200, overlap_chars=120
    ))

# ID 唯一且结果可重复；重新运行本单元会覆盖同一份本地索引输入。
assert len({chunk["chunk_id"] for chunk in all_chunks}) == len(all_chunks)
assert {chunk["language"] for chunk in all_chunks} == {"zh", "en"}
out_path = repo_root / "data/index/chunks.jsonl"
out_path.parent.mkdir(parents=True, exist_ok=True)
jsonl = "".join(json.dumps(chunk, ensure_ascii=False) + "\n" for chunk in all_chunks)
out_path.write_text(jsonl, encoding="utf-8")
print(f"已校验文件：{len(manifest)}；片段：{len(all_chunks)}")
print("语言分布：", dict(Counter(chunk["language"] for chunk in all_chunks)))
print("超过目标字符数的不可拆块：", sum(len(chunk["text"]) > 1200 for chunk in all_chunks))
print("输出：", out_path.relative_to(repo_root))
# 只预览自己生成的元数据，避免在 Notebook 输出中复制教材正文。
for chunk in all_chunks[:2]:
    print({key: chunk[key] for key in ("chunk_id", "language", "section", "line_start", "line_end", "url")})


## 下一步

请依次运行上面的单元，先看小样例断言，再查看中文与英文片段数量和出处链接。你可以在最后一个代码单元调整 max_chars 与 overlap_chars，比较片段粒度。当前输出是供后续索引使用的本地 JSONL；向量化、召回、重排和生成将在后续步骤实现。
